# Module 4 - LSTM Variant: SMILES Sequence Infilling

A self-contained **unidirectional LSTM** counterpart to the BiLSTM notebook. Everything lives in this notebook: no `config.py`, no `%%writefile`, no helper modules.

**Dataset:** [Clean QM9 Dataset for SeqLigase](https://www.kaggle.com/datasets/ssshaun/clean-qm9-dataset-fro-seqligase-project) - same 80/10/10 leakage-checked split, same tokenizer, same contiguous masking task.

**Kaggle setup**
1. Open this notebook on Kaggle.
2. Right sidebar -> **Accelerator** -> `GPU T4 x2` (one T4 is enough).
3. **Internet** -> On.
4. **Add Data** -> `Clean QM9 Dataset for SeqLigase Project`.

**What this notebook does**
- Downloads/attaches the dataset and builds the split and vocabulary in memory.
- Trains a unidirectional 3-layer LSTM with masked-token cross-entropy.
- Evaluates on the held-out test split (token accuracy, top-3, character accuracy, structural pass rate, RDKit validity).
- Saves the trained weights to `exports/seqlgase_lstm_best.pt` and exposes a download link.

**Architecture:** embedding 192 -> 3-layer unidirectional LSTM (hidden 384) -> LayerNorm -> vocabulary projection. Same optimized masking as the BiLSTM run: `max_mask_span=6`, `mask_span_power=4.0`, special-token logits banned.

**Expected runtime:** roughly 25-60 minutes for 30 epochs on a T4; early stopping often finishes sooner. For a smoke test, set `LSTMConfig(epochs=2, max_samples=5000)` in the training cell (add `max_samples` support if needed).

Run the cells top-to-bottom. Generated files land in `/kaggle/working`, where Kaggle exposes them as notebook output.


## 1. Environment setup

Everything this notebook needs is imported here. `rdkit` and `kagglehub` are installed only if missing (Kaggle normally ships both). Outputs are written to `exports/` and `plots/` under the current working directory.


In [ ]:
import importlib.util
import json
import math
import multiprocessing as mp
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path

for package, module in (("kagglehub", "kagglehub"), ("rdkit", "rdkit")):
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=False,
        )

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem, RDLogger
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

RDLogger.DisableLog("rdApp.*")

PROJECT_ROOT = Path.cwd()
EXPORT_DIR = PROJECT_ROOT / "exports"
PLOT_DIR = PROJECT_ROOT / "plots"
for directory in (EXPORT_DIR, PLOT_DIR, PROJECT_ROOT / "data" / "raw"):
    directory.mkdir(parents=True, exist_ok=True)

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Project root:", PROJECT_ROOT)


## 2. Download the dataset

Resolution order: existing CSVs in `data/raw/`, a dataset attached under `/kaggle/input`, `kagglehub.dataset_download(...)`, then the Kaggle CLI. Only CSV/JSON files are copied; the ECFP `.npy` arrays are not needed.


In [ ]:
KAGGLE_SLUG = "ssshaun/clean-qm9-dataset-fro-seqligase-project"
SMILES_COLUMNS = (
    "canonical_smiles",
    "smiles",
    "original_smiles",
    "smiles_string",
    "smi",
    "mol",
)


def _copy_tabular(source: Path, destination: Path) -> int:
    """Copy CSV/JSON files (skip the large ECFP .npy arrays)."""
    copied = 0
    for path in Path(source).rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".json"}:
            continue
        target = destination / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists() or target.stat().st_size != path.stat().st_size:
            shutil.copy2(path, target)
        copied += 1
    return copied


def download_dataset(destination: Path = PROJECT_ROOT / "data" / "raw") -> Path:
    """Resolve the dataset into data/raw using existing files, Kaggle input, kagglehub, or the CLI."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    if list(destination.rglob("*.csv")):
        print(f"[data] Using existing CSVs in {destination}")
        return destination

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.iterdir():
            name = candidate.name.lower()
            if "qm9" in name or "seqligase" in name:
                _copy_tabular(candidate, destination)
                if list(destination.rglob("*.csv")):
                    print(f"[data] Copied data from {candidate}")
                    return destination

    errors = []
    try:
        import kagglehub

        downloaded = Path(kagglehub.dataset_download(KAGGLE_SLUG))
        _copy_tabular(downloaded, destination)
        if list(destination.rglob("*.csv")):
            print(f"[data] Downloaded via kagglehub to {destination}")
            return destination
    except Exception as exc:
        errors.append(f"kagglehub: {exc}")

    try:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "kaggle",
                "datasets",
                "download",
                "-d",
                KAGGLE_SLUG,
                "-p",
                str(destination),
                "--unzip",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        if list(destination.rglob("*.csv")):
            print(f"[data] Downloaded via the Kaggle CLI to {destination}")
            return destination
    except Exception as exc:
        errors.append(f"kaggle CLI: {exc}")

    detail = "\n  ".join(errors) or "no download method available"
    raise RuntimeError(
        "Could not obtain the dataset. On Kaggle use Add Data and attach "
        f"'{KAGGLE_SLUG}'.\n  {detail}"
    )


DATA_ROOT = download_dataset()
print("Dataset root:", DATA_ROOT)
for path in sorted(DATA_ROOT.rglob("*.csv")):
    print(f"  {path.relative_to(DATA_ROOT)}  ({path.stat().st_size / 1e6:.1f} MB)")


## 3. Load, inspect, and split

The dataset already ships a leakage-checked 80/10/10 `split` column. `split_frame()` reuses it when present and otherwise performs a seeded two-stage scikit-learn split with `random_state=42`.


In [ ]:
TOKEN_PATTERN = re.compile(
    r"\[[^\]]+\]|Br|Cl|Si|Se|Na|Li|Mg|Ca|Al|@@|@|%\d{2}|.",
    re.DOTALL,
)


def tokenize_smiles(smiles: str) -> list[str]:
    """Split a SMILES string into chemical tokens."""
    if not isinstance(smiles, str):
        return []
    return TOKEN_PATTERN.findall(smiles.strip())


def identify_smiles_column(frame: pd.DataFrame) -> str:
    lower_to_original = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in SMILES_COLUMNS:
        if candidate in lower_to_original:
            return lower_to_original[candidate]
    for lowered, original in lower_to_original.items():
        if "smiles" in lowered:
            return original
    raise ValueError(f"Could not find a SMILES column in {list(frame.columns)}")


def load_smiles_frame(data_root: Path) -> pd.DataFrame:
    csv_files = sorted(Path(data_root).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under {data_root}")
    preferred = [path for path in csv_files if path.name.lower() == "qm9_clean.csv"]
    selected = preferred or csv_files

    frames = []
    for path in selected:
        frame = pd.read_csv(path)
        if frame.empty:
            continue
        column = identify_smiles_column(frame)
        frame = frame.copy()
        if column != "smiles":
            if "smiles" in frame.columns:
                frame = frame.drop(columns=["smiles"])
            frame = frame.rename(columns={column: "smiles"})
        keep = ["smiles"] + (["split"] if "split" in frame.columns else [])
        frames.append(frame[keep])

    if not frames:
        raise ValueError("No usable SMILES table was found in the dataset.")

    combined = pd.concat(frames, ignore_index=True).dropna(subset=["smiles"])
    combined["smiles"] = combined["smiles"].astype(str).str.strip()
    combined = combined[combined["smiles"] != ""].drop_duplicates("smiles", keep="first")
    return combined.reset_index(drop=True)


def split_frame(frame: pd.DataFrame, seed: int = 42) -> dict[str, pd.DataFrame]:
    """Reuse the dataset's leakage-checked split, else do a seeded 80/10/10 split."""
    if "split" in frame.columns:
        labels = frame["split"].astype(str).str.strip().str.lower()
        mapping = {
            "train": "train",
            "training": "train",
            "val": "val",
            "valid": "val",
            "validation": "val",
            "test": "test",
            "testing": "test",
        }
        mapped = labels.map(mapping)
        if mapped.notna().all() and {"train", "val", "test"}.issubset(mapped.unique()):
            parts = {
                name: frame.loc[mapped == name].reset_index(drop=True)
                for name in ("train", "val", "test")
            }
            if all(len(part) for part in parts.values()):
                return parts

    train_frame, remainder = train_test_split(
        frame, test_size=0.20, random_state=seed, shuffle=True
    )
    val_frame, test_frame = train_test_split(
        remainder, test_size=0.50, random_state=seed, shuffle=True
    )
    return {
        "train": train_frame.reset_index(drop=True),
        "val": val_frame.reset_index(drop=True),
        "test": test_frame.reset_index(drop=True),
    }


frame = load_smiles_frame(DATA_ROOT)
frame = frame.loc[frame["smiles"].map(lambda value: 1 <= len(tokenize_smiles(value)) <= 126)]
frame = frame.reset_index(drop=True)
print(f"Usable molecules: {len(frame):,}")
display(frame.head(8))

lengths = frame["smiles"].map(lambda value: len(tokenize_smiles(value)))
display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame("token_length").T)

SPLITS = split_frame(frame, seed=42)
for name, part in SPLITS.items():
    print(f"{name:>5}: {len(part):,} molecules ({100 * len(part) / len(frame):.1f}%)")


## 4. Vocabulary

The tokenizer keeps bracketed atoms (`[C@@H]`), two-letter elements, chirality markers, and ring-closure percentages as single tokens. The five special tokens occupy ids 0-4. The vocabulary is built from the training split only and saved to `exports/lstm_vocab.json`.


In [ ]:
SPECIAL_TOKENS = ("<PAD>", "<UNK>", "<MASK>", "<SOS>", "<EOS>")
PAD_TOKEN, UNK_TOKEN, MASK_TOKEN, SOS_TOKEN, EOS_TOKEN = SPECIAL_TOKENS
PAD_ID, UNK_ID, MASK_ID, SOS_ID, EOS_ID = range(len(SPECIAL_TOKENS))

IGNORE_INDEX = -100
# These tokens can never be a valid reconstruction, so their logits are banned.
OUTPUT_BANNED_IDS = (PAD_ID, UNK_ID, MASK_ID, SOS_ID, EOS_ID)


def build_vocab(smiles: list[str]) -> dict:
    """Build a deterministic vocabulary from the training split only."""
    counter: Counter[str] = Counter()
    for value in smiles:
        counter.update(tokenize_smiles(value))

    trained_tokens = sorted(
        (token for token in counter if token not in SPECIAL_TOKENS),
        key=lambda token: (-counter[token], token),
    )
    token_to_id = {token: index for index, token in enumerate(SPECIAL_TOKENS)}
    for token in trained_tokens:
        token_to_id[token] = len(token_to_id)
    id_to_token = [
        token for token, _ in sorted(token_to_id.items(), key=lambda item: item[1])
    ]
    return {
        "special_tokens": list(SPECIAL_TOKENS),
        "token_to_id": token_to_id,
        "id_to_token": id_to_token,
        "tokenizer_pattern": TOKEN_PATTERN.pattern,
        "num_tokens": len(id_to_token),
    }


VOCAB = build_vocab(SPLITS["train"]["smiles"].tolist())
(EXPORT_DIR / "lstm_vocab.json").write_text(
    json.dumps(VOCAB, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Vocabulary size : {VOCAB['num_tokens']:,}")
print(f"Special tokens  : {VOCAB['special_tokens']}")
print(f"First 50 tokens : {VOCAB['id_to_token'][:50]}")
print(f"Saved to        : {EXPORT_DIR / 'lstm_vocab.json'}")


## 5. Masking dataset

Each epoch the training dataset replaces a contiguous span with `<MASK>` and keeps the original ids as labels. `mask_span_power=4.0` biases spans toward short fragments (about 64% single-token masks), which is the main driver of token accuracy. Validation and test masks are fixed (`dynamic=False`).


In [ ]:
class MaskedSMILESDataset(Dataset):
    """Replace one contiguous token span with <MASK> and keep the originals as labels.

    ``mask_span_power`` biases the span distribution toward short fragments:
    ``span = 1 + int(max_span * u**power)``. With ``max_span=6`` and
    ``power=4.0`` roughly 64% of masks are a single token.
    """

    def __init__(
        self,
        smiles: list[str],
        vocab: dict,
        max_len: int = 128,
        max_mask_span: int = 6,
        mask_span_power: float = 4.0,
        seed: int = 42,
        dynamic: bool = True,
    ) -> None:
        self.smiles = [
            str(value).strip()
            for value in smiles
            if isinstance(value, str) and str(value).strip()
        ]
        if not self.smiles:
            raise ValueError("MaskedSMILESDataset received no valid SMILES.")
        self.token_to_id: dict[str, int] = vocab["token_to_id"]
        self.max_len = int(max_len)
        self.max_mask_span = int(max_mask_span)
        self.mask_span_power = max(float(mask_span_power), 1e-6)
        self.seed = int(seed)
        self.dynamic = bool(dynamic)
        self.epoch = 0

        self._cached_ids: list[list[int]] = []
        for value in self.smiles:
            tokens = tokenize_smiles(value)[: max(1, self.max_len - 2)]
            token_ids = [SOS_ID]
            token_ids.extend(self.token_to_id.get(token, UNK_ID) for token in tokens)
            token_ids.append(EOS_ID)
            self._cached_ids.append(token_ids)

    def __len__(self) -> int:
        return len(self.smiles)

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch) if self.dynamic else 0

    def _rng(self, index: int) -> random.Random:
        stream = (
            self.seed * 1_000_003 + index * 9_176 + self.epoch * 104_729
        ) % (2**32)
        return random.Random(stream)

    def __getitem__(self, index: int) -> dict:
        raw_smiles = self.smiles[index]
        token_ids = list(self._cached_ids[index])

        n_interior = len(token_ids) - 2
        if n_interior < 1:
            token_ids = [SOS_ID, UNK_ID, EOS_ID]
            n_interior = 1

        rng = self._rng(index)
        max_span = min(self.max_mask_span, max(1, n_interior - 1))
        span = 1 + int(max_span * (rng.random() ** self.mask_span_power))
        span = min(max(span, 1), max_span)

        start = rng.randint(1, n_interior - span + 1)
        end = start + span

        input_ids = list(token_ids)
        labels = [IGNORE_INDEX] * len(token_ids)
        for position in range(start, end):
            input_ids[position] = MASK_ID
            labels[position] = token_ids[position]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.ones(len(token_ids), dtype=torch.long),
            "original_ids": torch.tensor(token_ids, dtype=torch.long),
            "mask_start": start,
            "mask_end": end,
            "mask_length": span,
            "original_smiles": raw_smiles,
        }


def collate_masked_batch(batch: list[dict], pad_id: int = PAD_ID) -> dict:
    """Right-pad variable-length examples for the LSTM."""
    batch_size = len(batch)
    max_length = max(int(item["input_ids"].numel()) for item in batch)

    input_ids = torch.full((batch_size, max_length), pad_id, dtype=torch.long)
    labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_length), dtype=torch.long)
    original_ids = torch.full((batch_size, max_length), pad_id, dtype=torch.long)

    for row, item in enumerate(batch):
        length = int(item["input_ids"].numel())
        input_ids[row, :length] = item["input_ids"]
        labels[row, :length] = item["labels"]
        attention_mask[row, :length] = item["attention_mask"]
        original_ids[row, :length] = item["original_ids"]

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "original_ids": original_ids,
        "mask_start": torch.tensor([int(item["mask_start"]) for item in batch]),
        "mask_end": torch.tensor([int(item["mask_end"]) for item in batch]),
        "mask_length": torch.tensor([int(item["mask_length"]) for item in batch]),
        "original_smiles": [item["original_smiles"] for item in batch],
    }


preview = MaskedSMILESDataset(
    SPLITS["train"]["smiles"].tolist(),
    VOCAB,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    seed=42,
    dynamic=True,
)
sample = preview[0]
true_fragment = "".join(
    VOCAB["id_to_token"][int(token_id)]
    for token_id in sample["original_ids"][sample["mask_start"] : sample["mask_end"]]
)
print("Original        :", sample["original_smiles"])
print("Masked input    :", "".join(
    VOCAB["id_to_token"][int(token_id)]
    for token_id in sample["input_ids"]
    if int(token_id) != PAD_ID
))
print("Target fragment :", true_fragment)
print("Mask span       :", int(sample["mask_start"]), "->", int(sample["mask_end"]))


## 6. Unidirectional LSTM

`embed_dim=192` -> 3-layer unidirectional LSTM (`hidden_dim=384`) -> LayerNorm -> vocabulary projection. Padding uses `pack_padded_sequence` and the padding embedding is fixed at zero. This is the direct unidirectional counterpart to the BiLSTM notebook.


In [ ]:
class SeqLigaseLSTM(nn.Module):
    """Unidirectional LSTM encoder with a per-position vocabulary head.

    The model reads a (possibly masked) SMILES token sequence and predicts a
    distribution over the vocabulary at every position. Loss is applied only
    to masked positions, so the network learns to reconstruct the missing
    fragment from its left-to-right context.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 192,
        hidden_dim: int = 384,
        num_layers: int = 3,
        dropout: float = 0.2,
        pad_idx: int = PAD_ID,
    ) -> None:
        super().__init__()
        if vocab_size < 2:
            raise ValueError("vocab_size must be at least 2")
        if num_layers < 1:
            raise ValueError("num_layers must be at least 1")

        self.vocab_size = int(vocab_size)
        self.embed_dim = int(embed_dim)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.pad_idx = int(pad_idx)

        self.embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.embed_dim,
            padding_idx=self.pad_idx,
        )
        self.embed_dropout = nn.Dropout(self.dropout)
        self.lstm = nn.LSTM(
            input_size=self.embed_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=self.dropout if self.num_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(self.hidden_dim)
        self.output_dropout = nn.Dropout(self.dropout)
        self.output_projection = nn.Linear(self.hidden_dim, self.vocab_size)

        self.apply(self._init_weights)
        with torch.no_grad():
            self.embedding.weight[self.pad_idx].zero_()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Return logits of shape ``(batch, seq_len, vocab_size)``."""
        embedded = self.embed_dropout(self.embedding(input_ids))

        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1).clamp(min=1).to("cpu")
            packed = pack_padded_sequence(
                embedded,
                lengths,
                batch_first=True,
                enforce_sorted=False,
            )
            packed_outputs, _ = self.lstm(packed)
            outputs, _ = pad_packed_sequence(
                packed_outputs,
                batch_first=True,
                total_length=input_ids.size(1),
            )
        else:
            outputs, _ = self.lstm(embedded)

        outputs = self.layer_norm(outputs)
        return self.output_projection(self.output_dropout(outputs))

    @property
    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


preview_model = SeqLigaseLSTM(
    vocab_size=VOCAB["num_tokens"],
    embed_dim=192,
    hidden_dim=384,
    num_layers=3,
    dropout=0.2,
    pad_idx=PAD_ID,
)
print(preview_model)
print(f"Trainable parameters: {preview_model.num_parameters:,}")


## 7. Training utilities

Seeding, device selection, AMP, metric helpers, and the training epoch. Output logits for `<PAD>`, `<UNK>`, `<MASK>`, `<SOS>`, and `<EOS>` are restricted so the model cannot waste probability mass or win the argmax on structural tokens.


In [ ]:
def set_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def create_grad_scaler(enabled: bool, device_type: str):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def decode_ids(token_ids, id_to_token: list[str], skip_special: bool = True) -> str:
    tokens = []
    for raw_id in token_ids:
        index = int(raw_id)
        token = id_to_token[index] if 0 <= index < len(id_to_token) else UNK_TOKEN
        if skip_special and token in SPECIAL_TOKENS:
            continue
        tokens.append(token)
    return "".join(tokens)


def restrict_output_logits(logits: torch.Tensor) -> torch.Tensor:
    """Remove structural special tokens from the output distribution."""
    banned = torch.tensor(OUTPUT_BANNED_IDS, device=logits.device, dtype=torch.long)
    return logits.index_fill(-1, banned, -1e4)


def _token_statistics(logits: torch.Tensor, labels: torch.Tensor):
    mask = labels.ne(IGNORE_INDEX)
    n_tokens = int(mask.sum())
    if n_tokens == 0:
        empty = np.empty(0, dtype=np.int64)
        return 0, 0, 0, empty, empty

    predictions = logits.argmax(dim=-1)
    gold = labels[mask]
    guessed = predictions[mask]
    correct = int((guessed == gold).sum())

    top3_ids = logits.topk(3, dim=-1).indices
    top3_hits = (top3_ids == labels.unsqueeze(-1)).any(dim=-1) & mask
    top3_correct = int(top3_hits.sum())
    return (
        correct,
        top3_correct,
        n_tokens,
        gold.detach().cpu().numpy(),
        guessed.detach().cpu().numpy(),
    )


def _reconstruction_statistics(
    reconstructed: torch.Tensor,
    original_smiles: list[str],
    id_to_token: list[str],
    check_rdkit: bool,
) -> dict[str, float]:
    char_scores = []
    structural_passes = 0
    valid_predictions = 0
    n_sequences = 0

    for row, reference in zip(reconstructed, original_smiles):
        predicted_smiles = decode_ids(row.tolist(), id_to_token)
        char_scores.append(SequenceMatcher(None, predicted_smiles, reference).ratio())
        n_sequences += 1

        if check_rdkit:
            predicted_canonical = canonicalize(predicted_smiles)
            reference_canonical = canonicalize(reference)
            if predicted_canonical is not None:
                valid_predictions += 1
            if (
                predicted_canonical is not None
                and reference_canonical is not None
                and predicted_canonical == reference_canonical
            ):
                structural_passes += 1

    denominator = max(n_sequences, 1)
    return {
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "structural_pass_rate": structural_passes / denominator,
        "validity_rate": valid_predictions / denominator,
        "structural_passes": float(structural_passes),
        "valid_predictions": float(valid_predictions),
        "n_sequences": float(n_sequences),
    }


def canonicalize(smiles: str) -> str | None:
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    molecule = Chem.MolFromSmiles(smiles)
    return None if molecule is None else Chem.MolToSmiles(molecule)


def _macro_precision(gold_chunks: list[np.ndarray], predicted_chunks: list[np.ndarray]) -> float:
    if not gold_chunks or not predicted_chunks:
        return 0.0
    gold = np.concatenate(gold_chunks)
    predicted = np.concatenate(predicted_chunks)
    if gold.size == 0:
        return 0.0
    return float(precision_score(gold, predicted, average="macro", zero_division=0))


def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler,
    device: torch.device,
    config: "LSTMConfig",
    use_amp: bool,
    id_to_token: list[str],
) -> dict[str, float]:
    model.train()
    total_loss = 0.0
    total_tokens = 0
    correct = 0
    top3_correct = 0
    gold_chunks, predicted_chunks, char_scores = [], [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        original_ids = batch["original_ids"].to(device, non_blocking=True)

        n_tokens = int(labels.ne(IGNORE_INDEX).sum())
        if n_tokens == 0:
            continue

        optimizer.zero_grad(set_to_none=True)
        autocast_context = (
            torch.autocast(device_type=device.type, enabled=True)
            if use_amp
            else nullcontext()
        )
        with autocast_context:
            logits = restrict_output_logits(model(input_ids, attention_mask))
            loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.detach()) * n_tokens
        total_tokens += n_tokens

        with torch.no_grad():
            batch_correct, batch_top3, _, gold, guessed = _token_statistics(logits.detach(), labels)
            correct += batch_correct
            top3_correct += batch_top3
            gold_chunks.append(gold)
            predicted_chunks.append(guessed)

            mask = labels.ne(IGNORE_INDEX)
            reconstructed = torch.where(mask, logits.argmax(dim=-1), original_ids)
            reconstruction = _reconstruction_statistics(
                reconstructed, batch["original_smiles"], id_to_token, check_rdkit=False
            )
            char_scores.append(reconstruction["char_accuracy"])

    denominator = max(total_tokens, 1)
    return {
        "loss": total_loss / denominator if total_tokens else 0.0,
        "token_accuracy": correct / denominator if total_tokens else 0.0,
        "top3_accuracy": top3_correct / denominator if total_tokens else 0.0,
        "token_precision": _macro_precision(gold_chunks, predicted_chunks),
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "n_tokens": float(total_tokens),
    }


## 8. Evaluation and checkpoint utilities

Evaluation adds RDKit validity and structural reconstruction checks. `analyze_by_span_length()` reports token accuracy per mask length. `save_checkpoint()` writes the weights, model config, vocabulary, history, and metrics into one file.


In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool,
    id_to_token: list[str],
    check_rdkit: bool = True,
) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    correct = 0
    top3_correct = 0
    gold_chunks, predicted_chunks, char_scores = [], [], []
    structural_passes = 0
    valid_predictions = 0
    n_sequences = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        original_ids = batch["original_ids"].to(device, non_blocking=True)

        n_tokens = int(labels.ne(IGNORE_INDEX).sum())
        if n_tokens == 0:
            continue

        autocast_context = (
            torch.autocast(device_type=device.type, enabled=True)
            if use_amp
            else nullcontext()
        )
        with autocast_context:
            logits = restrict_output_logits(model(input_ids, attention_mask))
            loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

        total_loss += float(loss) * n_tokens
        total_tokens += n_tokens

        batch_correct, batch_top3, _, gold, guessed = _token_statistics(logits, labels)
        correct += batch_correct
        top3_correct += batch_top3
        gold_chunks.append(gold)
        predicted_chunks.append(guessed)

        mask = labels.ne(IGNORE_INDEX)
        reconstructed = torch.where(mask, logits.argmax(dim=-1), original_ids)
        reconstruction = _reconstruction_statistics(
            reconstructed, batch["original_smiles"], id_to_token, check_rdkit
        )
        char_scores.append(reconstruction["char_accuracy"])
        structural_passes += int(reconstruction["structural_passes"])
        valid_predictions += int(reconstruction["valid_predictions"])
        n_sequences += int(reconstruction["n_sequences"])

    denominator = max(total_tokens, 1)
    sequence_denominator = max(n_sequences, 1)
    return {
        "loss": total_loss / denominator if total_tokens else 0.0,
        "token_accuracy": correct / denominator if total_tokens else 0.0,
        "top3_accuracy": top3_correct / denominator if total_tokens else 0.0,
        "token_precision": _macro_precision(gold_chunks, predicted_chunks),
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "structural_pass_rate": structural_passes / sequence_denominator,
        "validity_rate": valid_predictions / sequence_denominator,
        "n_tokens": float(total_tokens),
        "n_sequences": float(n_sequences),
    }


@torch.no_grad()
def analyze_by_span_length(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    use_amp: bool,
) -> pd.DataFrame:
    """Token accuracy broken down by the number of masked tokens per example."""
    model.eval()
    totals: dict[int, dict[str, float]] = {}

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        span_lengths = batch["mask_length"].tolist()

        autocast_context = (
            torch.autocast(device_type=device.type, enabled=True)
            if use_amp
            else nullcontext()
        )
        with autocast_context:
            logits = restrict_output_logits(model(input_ids, attention_mask))

        predictions = logits.argmax(dim=-1)
        mask = labels.ne(IGNORE_INDEX)
        matches = (predictions == labels) & mask
        per_example_correct = matches.sum(dim=1).tolist()
        per_example_tokens = mask.sum(dim=1).tolist()

        for span, correct, tokens in zip(span_lengths, per_example_correct, per_example_tokens):
            entry = totals.setdefault(span, {"correct": 0.0, "tokens": 0.0, "examples": 0.0})
            entry["correct"] += float(correct)
            entry["tokens"] += float(tokens)
            entry["examples"] += 1.0

    rows = []
    for span in sorted(totals):
        entry = totals[span]
        rows.append(
            {
                "mask_length": int(span),
                "examples": int(entry["examples"]),
                "masked_tokens": int(entry["tokens"]),
                "token_accuracy": entry["correct"] / max(entry["tokens"], 1.0),
            }
        )
    return pd.DataFrame(rows)


def save_checkpoint(
    path: Path,
    model: nn.Module,
    config: "LSTMConfig",
    vocab: dict,
    epoch: int,
    best_val_loss: float,
    history: dict,
    test_metrics: dict | None = None,
) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "format_version": 1,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "vocab_size": len(vocab["token_to_id"]),
            "embed_dim": config.embed_dim,
            "hidden_dim": config.hidden_dim,
            "num_layers": config.num_layers,
            "dropout": config.dropout,
            "pad_idx": PAD_ID,
        },
        "train_config": config.to_dict(),
        "vocab": vocab,
        "epoch": int(epoch),
        "best_val_loss": float(best_val_loss),
        "history": json.loads(json.dumps(history)),
    }
    if test_metrics is not None:
        payload["test_metrics"] = test_metrics
    torch.save(payload, path)
    return path


def plot_training_curves(history: dict, output_dir: Path) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = history["epoch"]

    loss_figure, loss_axis = plt.subplots(figsize=(8, 5))
    loss_axis.plot(epochs, history["train_loss"], marker="o", linewidth=2, label="Train loss")
    loss_axis.plot(epochs, history["val_loss"], marker="s", linewidth=2, label="Validation loss")
    loss_axis.set_xlabel("Epoch")
    loss_axis.set_ylabel("Cross-entropy loss")
    loss_axis.set_title("SeqLigase LSTM - Loss Curve")
    loss_axis.grid(alpha=0.3)
    loss_axis.legend()
    loss_figure.tight_layout()
    loss_path = output_dir / "lstm_loss_curve.png"
    loss_figure.savefig(loss_path, dpi=200, bbox_inches="tight")
    plt.close(loss_figure)

    accuracy_figure, accuracy_axis = plt.subplots(figsize=(8, 5))
    accuracy_axis.plot(
        epochs, history["train_token_accuracy"], marker="o", linewidth=2,
        label="Train token accuracy",
    )
    accuracy_axis.plot(
        epochs, history["val_token_accuracy"], marker="s", linewidth=2,
        label="Validation token accuracy",
    )
    accuracy_axis.plot(
        epochs, history["val_top3_accuracy"], linestyle="--", linewidth=1.5,
        label="Validation top-3 accuracy",
    )
    accuracy_axis.set_xlabel("Epoch")
    accuracy_axis.set_ylabel("Accuracy")
    accuracy_axis.set_ylim(0.0, 1.0)
    accuracy_axis.set_title("SeqLigase LSTM - Accuracy Curve")
    accuracy_axis.grid(alpha=0.3)
    accuracy_axis.legend()
    accuracy_figure.tight_layout()
    accuracy_path = output_dir / "lstm_accuracy_curve.png"
    accuracy_figure.savefig(accuracy_path, dpi=200, bbox_inches="tight")
    plt.close(accuracy_figure)

    return {"loss_curve": loss_path, "accuracy_curve": accuracy_path}


def summary_dataframe(metrics: dict[str, float]) -> pd.DataFrame:
    rows = [
        ("Test Loss", f"{metrics['loss']:.4f}"),
        ("Test Accuracy (%)", f"{metrics['token_accuracy'] * 100:.2f}"),
        ("Top-3 Token Accuracy (%)", f"{metrics['top3_accuracy'] * 100:.2f}"),
        ("Character-Level Accuracy (%)", f"{metrics['char_accuracy'] * 100:.2f}"),
        ("Token Precision (macro)", f"{metrics['token_precision']:.4f}"),
        ("Structural Reconstruction Pass Rate (%)", f"{metrics['structural_pass_rate'] * 100:.2f}"),
        ("RDKit Validity Rate (%)", f"{metrics['validity_rate'] * 100:.2f}"),
        ("Evaluated Sequences", f"{int(metrics['n_sequences'])}"),
        ("Evaluated Masked Tokens", f"{int(metrics['n_tokens'])}"),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


## 9. Training loop

`fit()` builds the datasets/loaders, trains with AdamW + `ReduceLROnPlateau`, applies gradient clipping and early stopping, keeps the best checkpoint, then evaluates the held-out test split and saves every artefact.


In [ ]:
@dataclass
class LSTMConfig:
    """Hyper-parameters shared by training and evaluation."""

    seed: int = 42

    # Model
    embed_dim: int = 192
    hidden_dim: int = 384
    num_layers: int = 3
    dropout: float = 0.2

    # Sequence / masking
    max_len: int = 128
    max_mask_span: int = 6
    mask_span_power: float = 4.0

    # Optimization
    batch_size: int = 256
    epochs: int = 30
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    scheduler_factor: float = 0.5
    scheduler_patience: int = 3
    early_stopping_patience: int = 8

    # Runtime
    num_workers: int = 2
    amp: bool = True
    max_samples: int = 0  # 0 == use the full dataset

    def to_dict(self) -> dict:
        return asdict(self)


def fit(config: LSTMConfig, splits: dict[str, pd.DataFrame], vocab: dict) -> dict:
    """Train the LSTM, save the best weights, and evaluate on the test split."""
    ensure_dirs = (EXPORT_DIR, PLOT_DIR)
    for directory in ensure_dirs:
        directory.mkdir(parents=True, exist_ok=True)

    set_seed(config.seed)
    device = resolve_device()
    use_amp = bool(config.amp and device.type == "cuda")
    print(f"[device] {device} | AMP: {use_amp}")

    if config.max_samples > 0:
        caps = {
            "train": config.max_samples,
            "val": max(config.max_samples // 4, 200),
            "test": max(config.max_samples // 4, 200),
        }
        splits = {
            name: part.sample(n=min(len(part), caps[name]), random_state=config.seed).reset_index(drop=True)
            for name, part in splits.items()
        }
    print("[data] Split sizes - " + ", ".join(f"{name}: {len(part):,}" for name, part in splits.items()))

    datasets = {
        "train": MaskedSMILESDataset(
            splits["train"]["smiles"].tolist(), vocab,
            max_len=config.max_len, max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power, seed=config.seed, dynamic=True,
        ),
        "val": MaskedSMILESDataset(
            splits["val"]["smiles"].tolist(), vocab,
            max_len=config.max_len, max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power, seed=config.seed, dynamic=False,
        ),
        "test": MaskedSMILESDataset(
            splits["test"]["smiles"].tolist(), vocab,
            max_len=config.max_len, max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power, seed=config.seed, dynamic=False,
        ),
    }

    common = {
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "collate_fn": collate_masked_batch,
        "pin_memory": device.type == "cuda",
        "persistent_workers": config.num_workers > 0,
    }
    if config.num_workers > 0:
        # Python 3.14 defaults to "forkserver", which cannot re-import a
        # notebook kernel. "fork" works for Kaggle/Linux; fall back to
        # in-process loading where fork is unavailable (e.g. Windows).
        try:
            common["multiprocessing_context"] = mp.get_context("fork")
        except ValueError:
            common["num_workers"] = 0
            common["persistent_workers"] = False
    loaders = {
        "train": DataLoader(datasets["train"], shuffle=True, **common),
        "val": DataLoader(datasets["val"], shuffle=False, **common),
        "test": DataLoader(datasets["test"], shuffle=False, **common),
    }

    model = SeqLigaseLSTM(
        vocab_size=len(vocab["token_to_id"]),
        embed_dim=config.embed_dim,
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
        dropout=config.dropout,
        pad_idx=PAD_ID,
    ).to(device)
    print(f"[model] Trainable parameters: {model.num_parameters:,}")

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=config.scheduler_factor,
        patience=config.scheduler_patience, min_lr=1e-6,
    )
    scaler = create_grad_scaler(use_amp, device.type)
    id_to_token = vocab["id_to_token"]

    history = {
        "epoch": [], "train_loss": [], "val_loss": [],
        "train_token_accuracy": [], "val_token_accuracy": [],
        "train_top3_accuracy": [], "val_top3_accuracy": [],
        "train_char_accuracy": [], "val_char_accuracy": [],
        "train_token_precision": [], "val_token_precision": [],
        "val_structural_pass_rate": [], "val_validity_rate": [], "lr": [],
    }

    best_val_loss = math.inf
    best_epoch = 0
    epochs_without_improvement = 0
    best_path = EXPORT_DIR / "seqlgase_lstm_best.pt"
    last_path = EXPORT_DIR / "seqlgase_lstm_last.pt"

    for epoch in range(1, config.epochs + 1):
        datasets["train"].set_epoch(epoch)
        started_at = time.time()

        train_metrics = train_epoch(
            model, loaders["train"], criterion, optimizer, scaler,
            device, config, use_amp, id_to_token,
        )
        val_metrics = evaluate(
            model, loaders["val"], criterion, device, use_amp, id_to_token, check_rdkit=True
        )
        scheduler.step(val_metrics["loss"])
        current_lr = float(optimizer.param_groups[0]["lr"])
        elapsed = time.time() - started_at

        history["epoch"].append(epoch)
        for key in ("loss", "token_accuracy", "top3_accuracy", "char_accuracy", "token_precision"):
            history[f"train_{key}"].append(train_metrics[key])
            history[f"val_{key}"].append(val_metrics[key])
        history["val_structural_pass_rate"].append(val_metrics["structural_pass_rate"])
        history["val_validity_rate"].append(val_metrics["validity_rate"])
        history["lr"].append(current_lr)

        improved = val_metrics["loss"] < best_val_loss - 1e-6
        if improved:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            save_checkpoint(best_path, model, config, vocab, epoch, best_val_loss, history)
        else:
            epochs_without_improvement += 1
        save_checkpoint(last_path, model, config, vocab, epoch, best_val_loss, history)

        marker = " *best*" if improved else ""
        print(
            f"[epoch {epoch:02d}/{config.epochs:02d}] "
            f"loss {train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | "
            f"acc {train_metrics['token_accuracy'] * 100:.2f}/"
            f"{val_metrics['token_accuracy'] * 100:.2f} | "
            f"top3 {val_metrics['top3_accuracy'] * 100:.2f} | "
            f"char {val_metrics['char_accuracy'] * 100:.2f} | "
            f"prec {val_metrics['token_precision']:.3f} | "
            f"struct {val_metrics['structural_pass_rate'] * 100:.2f} | "
            f"lr {current_lr:.2e} | {elapsed:.1f}s{marker}"
        )

        if epochs_without_improvement >= config.early_stopping_patience:
            print(f"[early-stop] No validation improvement for {config.early_stopping_patience} epochs.")
            break

    print(f"[train] Best epoch: {best_epoch} | best val loss: {best_val_loss:.4f}")

    checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    best_model = SeqLigaseLSTM(**checkpoint["model_config"])
    best_model.load_state_dict(checkpoint["model_state_dict"])
    best_model.to(device)

    test_metrics = evaluate(
        best_model, loaders["test"], criterion, device, use_amp, id_to_token, check_rdkit=True
    )
    span_breakdown = analyze_by_span_length(best_model, loaders["test"], device, use_amp)
    plots = plot_training_curves(history, PLOT_DIR)

    checkpoint["test_metrics"] = test_metrics
    checkpoint["span_breakdown"] = span_breakdown.to_dict(orient="records")
    checkpoint["history"] = json.loads(json.dumps(history))
    torch.save(checkpoint, best_path)

    (EXPORT_DIR / "lstm_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    (EXPORT_DIR / "lstm_test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
    span_breakdown.to_csv(EXPORT_DIR / "lstm_span_length_metrics.csv", index=False)

    return {
        "history": history,
        "test_metrics": test_metrics,
        "span_breakdown": span_breakdown,
        "checkpoint": best_path,
        "plots": plots,
        "vocab_size": len(vocab["token_to_id"]),
    }


## 10. Train the model

Defaults match the optimized BiLSTM run: `max_mask_span=6`, `mask_span_power=4.0`, `embed_dim=192`, `hidden_dim=384`, `num_layers=3`, `scheduler_patience=3`, early stopping after 8 stagnant epochs.

> **Smoke test:** change `CONFIG` to `LSTMConfig(epochs=2, max_samples=5000, num_workers=2)`.


In [ ]:
CONFIG = LSTMConfig(
    seed=42,
    embed_dim=192,
    hidden_dim=384,
    num_layers=3,
    dropout=0.2,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    batch_size=256,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    scheduler_factor=0.5,
    scheduler_patience=3,
    early_stopping_patience=8,
    num_workers=2,
    amp=True,
    max_samples=0,
)

print(json.dumps(CONFIG.to_dict(), indent=2))
RESULT = fit(CONFIG, SPLITS, VOCAB)
print("\nBest checkpoint:", RESULT["checkpoint"])
print("Plot artefacts :", RESULT["plots"])


## 11. Training curves

Loss and accuracy curves are saved to `plots/lstm_loss_curve.png` and `plots/lstm_accuracy_curve.png`.


In [ ]:
from IPython.display import Image, display

for name, plot_path in RESULT["plots"].items():
    print(name, "->", plot_path)
    display(Image(filename=str(plot_path)))


## 12. Test metrics

Aggregate test metrics plus the per-span breakdown. The same values are saved to `exports/lstm_test_metrics.json` and `exports/lstm_span_length_metrics.csv`.


In [ ]:
print(f"Vocabulary size: {RESULT['vocab_size']:,} tokens")

display(summary_dataframe(RESULT["test_metrics"]).style.hide(axis="index"))

print("\nToken accuracy by masked-span length:")
display(
    RESULT["span_breakdown"]
    .style.format({"token_accuracy": "{:.2%}"})
    .hide(axis="index")
)


## 13. Save and download the weights

The best checkpoint contains `model_state_dict`, `model_config`, `vocab`, `train_config`, `history`, and `test_metrics`. A weights-only file is also written. Use the links below to download either file from Kaggle.


In [ ]:
from IPython.display import FileLink, display

BEST_PATH = Path(RESULT["checkpoint"])
checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)

state_dict_path = EXPORT_DIR / "seqlgase_lstm_state_dict.pt"
torch.save(checkpoint["model_state_dict"], state_dict_path)

print("Checkpoint:", BEST_PATH, f"({BEST_PATH.stat().st_size / 1e6:.2f} MB)")
print("Weights-only:", state_dict_path, f"({state_dict_path.stat().st_size / 1e6:.2f} MB)")
print("\nPayload keys:")
for key in checkpoint:
    print(f"  - {key}")
print("\nModel config:")
print(json.dumps(checkpoint["model_config"], indent=2))

for path in (BEST_PATH, state_dict_path):
    try:
        relative = path.relative_to(PROJECT_ROOT)
    except ValueError:
        relative = path
    display(FileLink(str(relative), result_html_prefix=f"Download {path.name}: "))


## 14. Verify the checkpoint

Rebuild the model from the saved config, load the state dict with `strict=True`, and run a dummy forward pass. This proves the exported file is usable outside the training loop.


In [ ]:
reloaded = SeqLigaseLSTM(**checkpoint["model_config"])
load_result = reloaded.load_state_dict(checkpoint["model_state_dict"], strict=True)
reloaded.eval()

assert reloaded.lstm.bidirectional is False, "This notebook must produce a unidirectional LSTM."
print("Missing keys   :", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Direction      :", "unidirectional" if not reloaded.lstm.bidirectional else "bidirectional")
print(f"Parameters     : {reloaded.num_parameters:,}")

dummy_ids = torch.tensor([[SOS_ID, 6, 7, MASK_ID, 8, EOS_ID], [SOS_ID, MASK_ID, EOS_ID, PAD_ID, PAD_ID, PAD_ID]])
dummy_mask = torch.tensor([[1, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0]])
with torch.no_grad():
    dummy_logits = reloaded(dummy_ids, dummy_mask)
print("Forward output :", tuple(dummy_logits.shape))
print("Checkpoint epoch:", checkpoint["epoch"])
print("Test token accuracy:", f"{checkpoint['test_metrics']['token_accuracy'] * 100:.2f}%")
print("RELOAD_OK")


## Notes and troubleshooting

**Generated files**
- `exports/seqlgase_lstm_best.pt` - best validation checkpoint (state dict + config + vocab + history + metrics)
- `exports/seqlgase_lstm_last.pt` - final-epoch checkpoint
- `exports/seqlgase_lstm_state_dict.pt` - raw weights only
- `exports/lstm_vocab.json`, `exports/lstm_history.json`, `exports/lstm_test_metrics.json`, `exports/lstm_span_length_metrics.csv`
- `plots/lstm_loss_curve.png`, `plots/lstm_accuracy_curve.png`

**Loading the weights elsewhere**

The checkpoint stores the state dict together with everything needed to rebuild the model, but the `SeqLigaseLSTM` class definition must be available. Copy the model cell into your project (or import it) and then:

```python
import torch

checkpoint = torch.load("exports/seqlgase_lstm_best.pt", map_location="cpu", weights_only=False)
model = SeqLigaseLSTM(**checkpoint["model_config"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(checkpoint["vocab"]["token_to_id"])
print(checkpoint["test_metrics"])
```

For an architecture-agnostic load, use `exports/seqlgase_lstm_state_dict.pt` and recreate `SeqLigaseLSTM` with `checkpoint["model_config"]`.

**Troubleshooting**

| Symptom | Fix |
|---|---|
| `Could not obtain the dataset` | Use **Add Data** on Kaggle and attach `ssshaun/clean-qm9-dataset-fro-seqligase-project`, or enable Internet so `kagglehub` can download it. |
| CUDA out of memory | Lower `batch_size` to 128, keep `max_len=128`. |
| Training is slow | Keep `num_workers=2` and confirm the accelerator is a GPU. AMP is already enabled. |
| `FileLink` does nothing | Run the cell in a live Kaggle session; the file is still in `exports/` and visible in the Output panel. |
| Want a quick run | `LSTMConfig(epochs=2, max_samples=5000)` still exercises every stage. |

**Metric definitions**
- **Token accuracy (top-1):** fraction of masked positions where the argmax matches the original token.
- **Top-3 token accuracy:** fraction where the original token is in the top three.
- **Character-level accuracy:** mean `difflib.SequenceMatcher` similarity between reconstructed and original SMILES.
- **Structural reconstruction pass rate:** RDKit-valid **and** canonically equal to the original.
- **RDKit validity rate:** RDKit can parse the reconstruction, regardless of whether it matches.

**Design differences from the BiLSTM notebook**
- The recurrent layer is unidirectional (`bidirectional=False`), so each prediction only sees the left context plus the mask itself.
- The notebook is fully self-contained: no generated `.py` modules and no OpenRouter/CLI/export code.
- Training hyper-parameters and the optimized masking distribution match the BiLSTM run, so the token-accuracy comparison is like-for-like.
